In [10]:


import os
import re
import sys

import pandas as pd

def read_data(path):
    # Read transfusion data
    pattern = re.compile(
        r".*/(?P<run_name>.+)/editing_eval/(?P<steps>.+)/(?P<task>.+)/bsz_(?P<b>.+)_ddpm_steps(?P<t>.+)_gs(?P<cfg>.+)_(?P<meta>.+)/t0_(?P<prompt>.+).jpg"
    )
    pattern = re.compile(
        r".*/(?P<run_name>.+)/editing_eval/(?P<steps>.+)/(?P<task>.+)_all_views_raw_renorm0.0_text(?P<cfg>.+)_img(?P<cfgimg>.+)_shift(?P<shift>.+)_nsteps(?P<nsteps>.+)_res(?P<res>.+)_heun(?P<heun>.+)_condition(?P<condition>.+)/(?P<exampleid>.+)/view(?P<view>.+)_(?P<type>.+).png"
    )
    pattern = re.compile(
        r".*/(?P<steps>.+)/(?P<task>.+)_all_views_raw_renorm0.0_text(?P<cfg>.+)_img(?P<cfgimg>.+)_shift(?P<shift>.+)_nsteps(?P<nsteps>.+)_res(?P<res>.+)_heun(?P<heun>.+)_condition(?P<condition>.+)/(?P<exampleid>.+)/view(?P<view>.+)_(?P<type>.+).png"
    )
    #g1h1_drawer_rollout_all_views_raw_renorm0.0_text4.0_img1.2_shift3.0_nsteps25_res224_heunFalse_conditionFalse_
    # Extracting metadata from files
    data = []
    print(path)
    for file in list_files_recursive(path):
        print(file)
        match = pattern.match(file)
        if match:
            metadata = match.groupdict()
            metadata["path"] = file
            data.append(metadata)

    # Read transfusion data
    # #/checkpoint/onellm/liliyu/expriments/cm3v2/v2760m_0.5T_qk_zloss_linear_zero3_b4/v2760m_0.5T_qk_zloss_linear_zero3_b4_run000/gen/checkpoint_0010000/gen_size_512/cfg5.0_t0.7_p0.9/t0_dog_1.jpg
    # pattern = re.compile(
    #     r".*/(?P<run_name>.+)/*/gen/checkpoint_(?P<steps>.+)/(?P<task>.+)/cfg(?P<cfg>.+)_(?P<meta>.+)/t0_(?P<prompt>.+).jpg"
    # )
    # #/home/liliyu/rsc/gen_ms/760mxlfdiff_0.5T_256_sstkllama_latentfl8_run000/gen/checkpoint_0020000/gen_size_256/bsz_12_ddpm_steps250_gs5.0_grs0.0_dythres0.995/t0_dog_1.jpg
    # # Extracting metadata from files
    # for file in list_files_recursive(path):
    #     if "xlfdiff" in file:
    #         continue

    #     match = pattern.match(file)
    #     # print(file)
    #     # print(match)
    #     if match:
    #         metadata = match.groupdict()
    #         metadata['t'] = 0
    #         metadata['b'] = 0
    #         metadata["path"] = file
    #         data.append(metadata)

    # Creating DataFrame
    data_df = pd.DataFrame(data)
    return data_df


def list_files_recursive(path):
    for root, _, files in os.walk(path):
        for file in files:
            yield os.path.join(root, file)


# if __name__ == "__main__":
#     path = "/home/liliyu/workspace/BAGEL/results/h1g1_vit_t1.0_gpu16_seq16384_shard8\=_/editing_eval"
#     main(path)


In [13]:
path= "/home/liliyu/workspace/BAGEL/results/h1g1_vit_t1.0_gpu16_seq16384_shard8\=_/editing_eval"
data= read_data(path)

/home/liliyu/workspace/BAGEL/results/h1g1_vit_t1.0_gpu16_seq16384_shard8\=_/editing_eval


In [1]:
import re
import streamlit as st
import pandas as pd
from pathlib import Path
from PIL import Image
import json
import pickle
from datetime import datetime, timedelta
import tqdm

In [5]:
# def scan_results_directory(base_path="/home/liliyu/workspace/BAGEL/results", force_refresh=False):
#     """Scan the results directory to find all available models and checkpoints"""
    
# Try to load from cache first if not forcing refresh
base_path="/home/liliyu/workspace/BAGEL/results"
data = []
base_path = Path(base_path)

# if not base_path.exists():
#     st.error(f"Results directory not found: {base_path}")
#     return pd.DataFrame()

# Pattern for parsing hyperparameter folders
hyperparam_pattern = re.compile(
    r"(?P<task>[^_]+_[^_]+(?:_[^_]+)?)_all_views_raw_renorm(?P<renorm>[\d.]+)_text(?P<textcfg>[\d.]+)_img(?P<imgcfg>[\d.]+)_shift(?P<shift>[\d.]+)_nsteps(?P<nsteps>\d+)_res(?P<res>\d+)_heun(?P<heun>\w+)_condition(?P<condition>\w+)_?"
)

# Count total directories for progress bar
run_dirs = [d for d in base_path.iterdir() if d.is_dir() and (d.name.startswith("seed") or "h1g1" in d.name)]
print(run_dirs)
total_runs = len(run_dirs)

# # Create progress container
# progress_container = st.container()
# with progress_container:
#     progress_bar = st.progress(0)
#     status_text = st.empty()

# Scan all directories
for idx, run_dir in tqdm.tqdm(enumerate(run_dirs)):
    # Update progress
    # progress = (idx + 1) / total_runs
    # progress_bar.progress(progress)
    # status_text.text(f"Scanning: {run_dir.name} ({idx+1}/{total_runs})")
    
    editing_eval_path = run_dir / "editing_eval"
    if not editing_eval_path.exists():
        continue
        
    run_name = run_dir.name
    
    # Scan checkpoints
    for checkpoint_dir in editing_eval_path.iterdir():
        if not checkpoint_dir.is_dir():
            continue
            
        checkpoint_steps = checkpoint_dir.name
        
        # Scan hyperparameter directories
        for hyperparam_dir in checkpoint_dir.iterdir():
            if not hyperparam_dir.is_dir():
                continue
            
            print('trying to match', hyperparam_dir.name)
            match = hyperparam_pattern.match(hyperparam_dir.name)
            if match:
                print('matched', match.groupdict())
                info = match.groupdict()
                info['run_name'] = run_name
                info['checkpoint_steps'] = checkpoint_steps
                info['hyperparam_dir'] = hyperparam_dir.name
                info['full_path'] = str(hyperparam_dir)
                
                # Count available examples (faster method)
                try:
                    example_count = len([d for d in hyperparam_dir.iterdir() if d.is_dir()])
                except Exception:
                    example_count = 0
                info['example_count'] = example_count
                
                data.append(info)

df = pd.DataFrame(data)


[PosixPath('/home/liliyu/workspace/BAGEL/results/pi_h1g1_allview_seq_seedp1_gpu16_seq16384'), PosixPath('/home/liliyu/workspace/BAGEL/results/seedp1_0.2_mm_statics_text_allview_endspan_gpu32_seq16384'), PosixPath('/home/liliyu/workspace/BAGEL/results/seedp1_0.2_mm_statics_text_allview_endspan_heuristic_gpu32_seq16384'), PosixPath('/home/liliyu/workspace/BAGEL/results/seedp1_0.2_all_robots_july17_gpu64_seq16384'), PosixPath('/home/liliyu/workspace/BAGEL/results/seedp1_0.2_all_robots_july17_gpu16seq16384'), PosixPath('/home/liliyu/workspace/BAGEL/results/seedp1_0.2_all_robots_july17_gpu16_seq16384'), PosixPath('/home/liliyu/workspace/BAGEL/results/seed_blip30_all_robots_july17_tt1'), PosixPath('/home/liliyu/workspace/BAGEL/results/seed_blip30_all_robots_july17_tt3'), PosixPath('/home/liliyu/workspace/BAGEL/results/seed_blip30_all_robots_july17_tt4'), PosixPath('/home/liliyu/workspace/BAGEL/results/seed_blip30_all_robots_july17_tt0'), PosixPath('/home/liliyu/workspace/BAGEL/results/seed_b

58it [00:00, 570.36it/s]

84it [00:00, 733.07it/s]

trying to match g1h1_endspan_all_views_raw_renorm0.0_text4.0_img1.2_shift3.0_res224
trying to match g1h1_endspan_all_views_raw_renorm0.0_text4.0_img1.2_shift3.0_nsteps25_res224
trying to match g1h1_endspan_all_views_raw_renorm0.0_text4.0_img1.2_shift3.0_nsteps20_res224
trying to match g1h1_endspan_all_views_raw_renorm0.0_text4.0_img1.2_shift3.0_nsteps10_res224_heunTrue
trying to match g1h1_endspan_all_views_raw_renorm0.0_text4.0_img1.2_shift1.0_nsteps10_res224_heunTrue
trying to match g1h1_endspan_all_views_raw_renorm0.0_text4.0_img1.2_shift3.0_nsteps20_res224_heunTrue
trying to match g1h1_endspan_all_views_raw_renorm0.0_text4.0_img1.2_shift3.0_nsteps25_res224_heunFalse
trying to match g1h1_endspan_1_all_views_raw_renorm0.0_text4.0_img1.2_shift3.0_nsteps25_res224_heunFalse_conditionFalse_
matched {'task': 'g1h1_endspan_1', 'renorm': '0.0', 'textcfg': '4.0', 'imgcfg': '1.2', 'shift': '3.0', 'nsteps': '25', 'res': '224', 'heun': 'False', 'condition': 'False_'}
trying to match h1g1_dishes

In [6]:
df


,task,renorm,textcfg,imgcfg,shift,nsteps,res,heun,condition,run_name,checkpoint_steps,hyperparam_dir,full_path,example_count
0,g1h1_endspan_1,0.0,4.0,1.2,3.0,25,224,False,False_,pi_h1g1_allview_seq_seedp1_gpu16_seq16384,0070000,g1h1_endspan_1_all_views_raw_renorm0.0_text4.0...,/home/liliyu/workspace/BAGEL/results/pi_h1g1_a...,100
1,g1h1_endspan_1,0.0,4.0,1.2,3.0,25,224,False,False_,pi_h1g1_allview_seq_seedp1_gpu16_seq16384,0050000,g1h1_endspan_1_all_views_raw_renorm0.0_text4.0...,/home/liliyu/workspace/BAGEL/results/pi_h1g1_a...,100
2,g1h1_drawer_rollout,0.0,4.0,1.2,3.0,50,224,False,False_,seed_blip3o_all_robots_jul19_gpu64seq16384_pre...,0013000,g1h1_drawer_rollout_all_views_raw_renorm0.0_te...,/home/liliyu/workspace/BAGEL/results/seed_blip...,20
3,g1h1_drawer_rollout,0.0,4.0,1.2,3.0,50,224,False,False_,seed_blip3o_all_robots_jul19_gpu64seq16384_pre...,0020000,g1h1_drawer_rollout_all_views_raw_renorm0.0_te...,/home/liliyu/workspace/BAGEL/results/seed_blip...,20
4,g1h1_drawer_rollout,0.0,4.0,1.2,1.0,50,224,False,False_,seed_blip3o_all_robots_jul19_gpu64seq16384_pre...,0020000,g1h1_drawer_rollout_all_views_raw_renorm0.0_te...,/home/liliyu/workspace/BAGEL/results/seed_blip...,20
5,g1h1_drawer_rollout,0.0,4.0,1.2,3.0,25,224,False,False_,seed_blip3o_all_robots_jul19_gpu64_seq16384_sh...,0014000,g1h1_drawer_rollout_all_views_raw_renorm0.0_te...,/home/liliyu/workspace/BAGEL/results/seed_blip...,20
6,g1h1_drawer_rollout,0.0,4.0,1.2,3.0,25,224,False,False_,seed_blip3o_all_robots_jul19_gpu64_seq16384_sh...,0024000,g1h1_drawer_rollout_all_views_raw_renorm0.0_te...,/home/liliyu/workspace/BAGEL/results/seed_blip...,20
7,g1h1_drawer_rollout,0.0,4.0,1.2,3.0,25,224,False,False_,seed_blip3o_all_robots_jul19_gpu64_seq16384_sh...,0040000,g1h1_drawer_rollout_all_views_raw_renorm0.0_te...,/home/liliyu/workspace/BAGEL/results/seed_blip...,20
8,g1h1_drawer_rollout,0.0,4.0,1.2,1.0,25,224,False,False_,seed_blip3o_all_robots_jul19_gpu64_seq16384_sh...,0010000,g1h1_drawer_rollout_all_views_raw_renorm0.0_te...,/home/liliyu/workspace/BAGEL/results/seed_blip...,20
9,g1h1_drawer_rollout,0.0,4.0,1.2,1.0,25,224,False,False_,seed_blip3o_all_robots_jul19_t1.0_gpu16_seq163...,0100000,g1h1_drawer_rollout_all_views_raw_renorm0.0_te...,/home/liliyu/workspace/BAGEL/results/seed_blip...,20
